# Μάθημα 13 - Μνήμη Πράκτορα με Γραφήματα Γνώσης Cognee


## Ρυθμίσεις

Αυτό το σημειωματάριο δείχνει πώς να δημιουργήσετε έναν έξυπνο **βοηθό κωδικοποίησης** με επίμονη μνήμη χρησιμοποιώντας τα γραφήματα γνώσης [**Cognee**](https://www.cognee.ai/) και το **Microsoft Agent Framework** (MAF).

Το Cognee μετατρέπει το μη δομημένο κείμενο σε ένα δομημένο, ερωτήσιμο γράφημα γνώσης υποστηριζόμενο από ενσωματώσεις διάνυσμάτων — δίνοντας στον πράκτορά σας μια πλούσια μνήμη μακράς διαρκείας με επίγνωση σχέσεων.

### Τι θα μάθετε
1. **Δημιουργία Γραφημάτων Γνώσης**: Μετατρέψτε προφίλ προγραμματιστών και βέλτιστες πρακτικές σε δομημένη, ερωτήσιμη γνώση.
2. **Ενσωμάτωση του Cognee με το MAF**: Χρησιμοποιήστε τις συναρτήσεις `@tool` για να επιτρέψετε σε έναν πράκτορα MAF να ερωτά το γράφημα γνώσης του Cognee.
3. **Συνομιλίες με επίγνωση συνεδρίας**: Διατηρήστε το πλαίσιο ανάμεσα σε πολλαπλές ερωτήσεις στην ίδια συνεδρία.
4. **Μνήμη μακράς διαρκείας**: Διατηρήστε σημαντική γνώση ανάμεσα σε συνεδρίες και ανακτήστε την σε νέες συνομιλίες.

### Απαιτήσεις
- Python 3.9+
- Redis που τρέχει τοπικά (`docker run -d -p 6379:6379 redis`) για διαχείριση συνεδριών
- Ένα κλειδί API LLM (π.χ. OpenAI) — ορίστε `LLM_API_KEY` στο `.env`
- `CACHING=true` στο `.env` (απαραίτητο για συνεδρίες Cognee)
- Ένα έργο Microsoft Foundry με αναπτυγμένο μοντέλο συνομιλίας
- `AZURE_AI_PROJECT_ENDPOINT` και `AZURE_AI_MODEL_DEPLOYMENT_NAME` στο `.env`
- Είσοδος με Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Τύποι Μνήμης Πράκτορα

Αυτό το σημειωματάριο εξερευνά τους ίδιους τρεις τύπους μνήμης από το κύριο σημειωματάριο του Μαθήματος 13, αλλά χρησιμοποιεί το Cognee ως το backend μακροπρόθεσμης μνήμης:

| Τύπος Μνήμης | Μηχανισμός | Διάρκεια Ζωής |
|---|---|---|
| **Εργασίας** | `agent.create_session()` (MAF) | Μία συνομιλία |
| **Βραχυπρόθεσμη** | Cache συνεδρίας Cognee (Redis) | Μία συνεδρία |
| **Μακροπρόθεσμη** | Γραφική γνώσης Cognee + διανύσματα | Μόνιμη |

### Αρχιτεκτονική Μνήμης του Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Προετοιμασία Αποθήκευσης Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Μέρος 1 — Δημιουργία της Βάσης Γνώσεων

Εισάγουμε τρεις τύπους δεδομένων για να δημιουργήσουμε μια ολοκληρωμένη βάση γνώσεων για τον βοηθό προγραμματισμού μας:

1. **Προφίλ Προγραμματιστή** — προσωπική εμπειρογνωμοσύνη και τεχνικό υπόβαθρο
2. **Καλές Πρακτικές Python** — το Ζεν της Python με πρακτικές κατευθυντήριες γραμμές
3. **Ιστορικές Συζητήσεις** — προηγούμενες συνεδρίες ερωτήσεων και απαντήσεων μεταξύ προγραμματιστών και βοηθών AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Οπτικοποιήστε το Γράφημα Γνώσης

Το Cognee μπορεί να δημιουργήσει μια διαδραστική απεικόνιση HTML των οντοτήτων και των σχέσεων που εξήγαγε.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Εμπλουτίστε τη Μνήμη με το Memify

Το `memify()` αναλύει το γράφημα γνώσης και δημιουργεί ευφυείς κανόνες — εντοπίζοντας πρότυπα, βέλτιστες πρακτικές και σχέσεις μεταξύ εννοιών.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Μέρος 2 — Πράκτορας MAF με τα Εργαλεία Cognee

Τώρα δημιουργούμε έναν πράκτορα MAF που μπορεί να κάνει ερωτήματα στο γράφο γνώσης του Cognee μέσω συναρτήσεων `@tool`. Αυτό επιτρέπει στον πράκτορα να αξιοποιεί όλη τη δύναμη της σημασιολογικής αναζήτησης με επίγνωση γραφήματος, διατηρώντας παράλληλα το συμφραζόμενο της συνομιλίας μέσα από συνεδρίες.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Εργαζόμενη Μνήμη με Συνεδρίες

Το `AgentSession` (δημιουργείται μέσω του `agent.create_session()`) παρέχει εργαζόμενη μνήμη μέσα σε μια συνεδρία. Ο πράκτορας μπορεί να ανατρέξει σε προηγούμενα μηνύματα ενώ ταυτόχρονα συμβουλεύεται το μακροπρόθεσμο γράφημα γνώσης του Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Νέα Συνεδρία — Η Μακροπρόθεσμη Μνήμη Παραμένει

Η έναρξη μιας νέας συνεδρίας καθαρίζει τη βραχυπρόθεσμη μνήμη, αλλά ο γράφος γνώσεων Cognee είναι ακόμα διαθέσιμος. Ο πράκτορας μπορεί να ανακτήσει την ίδια μακροπρόθεσμη γνώση σε μια ολοκαίνουργια συνομιλία.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Περίληψη

Σε αυτό το σημειωματάριο δημιουργήσατε έναν βοηθό κωδικοποίησης που συνδυάζει τη **βραχυπρόθεσμη μνήμη του MAF** (`agent.create_session()`) με το **μακροπρόθεσμο γράφο γνώσης του Cognee**.

### Τι μάθατε
1. **Κατασκευή γράφου γνώσης**: Το Cognee επεξεργάζεται μη δομημένο κείμενο και δημιουργεί έναν γράφο + διανυσματική μνήμη.
2. **Εμπλουτισμός γράφου με memify**: Παράγωγα δεδομένα και πλουσιότερες σχέσεις πάνω στον υπάρχοντα γράφο σας.
3. **Ενσωμάτωση MAF + Cognee**: Οι λειτουργίες `@tool` επιτρέπουν σε έναν πράκτορα MAF να ερωτά τον γράφο του Cognee με φυσικό τρόπο.
4. **Βραχυπρόθεσμη + μακροπρόθεσμη μνήμη**: Το `AgentSession` (μέσω `agent.create_session()`) παρέχει πλαίσιο συνεδρίας ενώ το Cognee παρέχει μόνιμη γνώση.
5. **Φιλτραρισμένη αναζήτηση με NodeSets**: Εστίαση σε συγκεκριμένα υποσύνολα του γράφου γνώσης (π.χ. μόνο αρχές).

### Κύρια σημεία
- Το **Cognee** μετατρέπει το ακατέργαστο κείμενο σε δομημένη, σχέσεων-ενήμερη μνήμη — πιο ισχυρή από μια απλή διανυσματική αποθήκη.
- Οι **λειτουργίες `@tool`** γεφυρώνουν τους πράκτορες MAF με εξωτερικά συστήματα γνώσης καθαρά.
- Το **`AgentSession`** (μέσω `agent.create_session()`) κρατά το πλαίσιο συζήτησης ξεχωριστό από τη μακροχρόνια γνώση.
- Ο ίδιος γράφος γνώσης εξυπηρετεί πολλαπλές συνεδρίες και πράκτορες.

### Εφαρμογές στην πράξη
- **Προγραμματιστές συνοδηγοί**: Ανασκόπηση κώδικα, ανάλυση συμβάντων, βοηθοί αρχιτεκτονικής
- **Συνοδηγοί εξυπηρέτησης πελατών**: Υποστηρικτικοί πράκτορες σε έγγραφα προϊόντων, συχνές ερωτήσεις και σημειώσεις CRM
- **Εσωτερικοί ειδικοί συνοδηγοί**: Βοηθοί πολιτικής, νομικοί ή ασφαλείας που λογικοποιούν κανονισμούς
- **Ενοποιημένα επίπεδα δεδομένων**: Συνδυασμός δομημένων και μη δομημένων δεδομένων σε έναν γράφο αναζητήσιμο

### Επόμενα βήματα
- Πειραματιστείτε με χρονική επίγνωση στο Cognee
- Ορίστε μια οντολογία OWL για την ποιότητα γράφων σε συγκεκριμένο τομέα
- Προσθέστε βρόχους ανατροφοδότησης χρηστών για βελτίωση της ανάκτησης με την πάροδο του χρόνου
- Κλιμακώστε σε συστήματα με πολλαπλούς πράκτορες που μοιράζονται το ίδιο επίπεδο μνήμης Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
